# Endpoint Ollama pe GPU (model pentru demo)
Pornește `ollama serve` cu **qwen2.5:14b-instruct-q4_K_M** și îl expune public printr-un tunel **cloudflared** (URL `https://*.trycloudflare.com`).

**Setup:** Accelerator = GPU **T4** (un singur GPU ajunge pt 14B-Q4 ~9GB), Internet = **On**.
Rulează celulele 1→4. Copiază URL-ul din celula 4 și pornește local:
`OLLAMA_HOST=<url> ./scripts/demo_remote.sh`

Pt 7B (mai ieftin/rapid): schimbă tag-ul în `qwen2.5:7b-instruct-q4_K_M`.

In [ ]:
# 1. Instalează zstd (necesar pt extracția ollama), apoi ollama + cloudflared
!sudo apt-get install -y zstd >/dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
import shutil; print('ollama:', shutil.which('ollama'), '| cloudflared:', shutil.which('cloudflared'))

In [ ]:
# 2. Pornește ollama serve (cwd stabil /root ca să nu cadă runner-ul)
import subprocess, time, os, urllib.request
subprocess.run(['pkill','-f','ollama serve'], capture_output=True); time.sleep(2)
log=open('/tmp/ollama.log','w')
subprocess.Popen(['ollama','serve'], stdout=log, stderr=log, cwd='/root')
for _ in range(60):
    try: urllib.request.urlopen('http://127.0.0.1:11434/api/version',timeout=2); print('ollama UP'); break
    except Exception: time.sleep(1)
else: raise RuntimeError('ollama down')

In [ ]:
# 3. Pull + warm modelul demo (14B-Q4 ~9GB). Warm via CLI (nu e nevoie de libul python ollama).
MODEL='qwen2.5:14b-instruct-q4_K_M'
import subprocess
subprocess.run(['ollama','pull',MODEL], check=True)
subprocess.run(['ollama','run',MODEL,'Salut, ce faci?'], check=True)  # încarcă în VRAM
print('gata:', MODEL)

In [ ]:
# 4. Tunel public (în BACKGROUND, ca să nu blocheze kernelul) + auto-detectează URL + self-test.
import subprocess, time, re, urllib.request

# (a) ollama local viu?
try:
    v=urllib.request.urlopen('http://localhost:11434/api/version', timeout=5).read().decode()
    print('local ollama OK:', v)
except Exception as e:
    print('LOCAL OLLAMA DOWN ->', e, '\n   re-rulează celula 2 (ollama serve) și 3 (pull).')

# (b) pornește cloudflared în background
subprocess.run(['pkill','-f','cloudflared'], capture_output=True); time.sleep(1)
LOG='/tmp/cf.log'
subprocess.Popen(['cloudflared','tunnel','--url','http://localhost:11434',
                  '--http-host-header','localhost:11434'],  # ollama blochează Host != localhost (403)
                 stdout=open(LOG,'w'), stderr=subprocess.STDOUT)

# (c) extrage URL-ul public din log
URL=None
for _ in range(40):
    time.sleep(1)
    try: m=re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', open(LOG).read())
    except FileNotFoundError: m=None
    if m: URL=m.group(0); break
print('\nTUNNEL URL:', URL)

# (d) self-test PRIN tunelul public (tunnel -> ollama). Dacă merge aici, merge și pe laptop.
ok=False
for _ in range(20):
    try:
        r=urllib.request.urlopen(URL+'/api/version', timeout=15).read().decode()
        print('PUBLIC ENDPOINT OK:', r); ok=True; break
    except Exception as e:
        print('  aștept tunelul...', type(e).__name__); time.sleep(3)
if ok:
    print('\n>>> Rulează local:\n    OLLAMA_HOST=%s ./scripts/demo_remote.sh' % URL)
else:
    print('\n>>> Tunel/ollama indisponibil. Vezi /tmp/cf.log:'); print(open(LOG).read()[-800:])